In [ ]:
import requests
import json

# Download the data
resp = requests.get('https://raw.githubusercontent.com/weaviate-tutorials/quickstart/main/data/jeopardy_tiny.json')
data = json.loads(resp.text)  # Load data
    
def json_print(data):
    print(json.dumps(data, indent=2))
    
json_print(data)

In [ ]:
import weaviate
import weaviate.classes as wvc
import os

client = weaviate.connect_to_local(headers={"X-OpenAI-Api-Key": os.environ["OPENAI_API_KEY"]})

In [ ]:
client.collections.delete("Question")

In [ ]:
questions = client.collections.create("Question",
    vectorizer_config=wvc.config.Configure.Vectorizer.text2vec_openai()
)

In [ ]:
questions.data.insert_many(data)

In [ ]:
response = questions.aggregate.over_all(total_count=True)
print(response.total_count)

### Lets perform vector search for the concept of "animal"

In [ ]:
#Vector search: a query that will try to match concepts between the query and objects

response = questions.query.near_text(query="animals").with_limit(2).do()

print(json.dumps(response, indent=4))

### Now, lets perform keyword search

In [ ]:
#Write a query that will try to match the words in the query to the words in the object

response = questions.query.bm25(query="animals").with_limit(2).do()

print(json.dumps(response, indent=4))

### Why do we only get one match here? We know there are more animal related objects!

### Lets combine keyword and vector search - called hybrid search!

In [ ]:
#Hybrid search query

response = questions.query.hybrid(query="animals").with_limit(5).do()

print(json.dumps(response, indent=4))

In [ ]:
#Modify the alpha parameter to see what happens!

response = questions.query.hybrid(query="animals", alpha=0.5).with_limit(5).do()

print(json.dumps(response, indent=4))

### Notice the order of the returned results!